# Publication-Quality Figures for Report
Generate all figures needed for the 20-page competition report.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from hdi.config import *
from hdi.data.loaders import load_panel
from hdi.visualization.charts import (
    plot_choropleth, plot_lisa_map, plot_line_chart,
    plot_stacked_bar, plot_scatter, plot_bump_chart,
    plot_bivariate_choropleth, plot_quadrant_scatter,
)
from hdi.visualization.interactive import (
    interactive_sankey, interactive_ghri_radar,
    interactive_scenario_fan,
)
from hdi.visualization.shap_plots import plot_shap_beeswarm
from hdi.features.indices import compute_ghri


def save_figure(fig, name, dpi=300, formats=("png", "pdf")):
    """Save figure to reports/figures/ in multiple formats."""
    out_dir = Path("..") / "reports" / "figures"
    out_dir.mkdir(parents=True, exist_ok=True)
    for fmt in formats:
        path = out_dir / f"{name}.{fmt}"
        fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
        print(f"  Saved: {path}")


panel = load_panel()
print(f"Panel shape: {panel.shape}")
print(f"Years: {panel['year'].min()} - {panel['year'].max()}")
print(f"Countries: {panel['country'].nunique()}")

## Figure 1: DALY Rate World Map (Latest Year)

In [ ]:
# Figure 1: Choropleth of age-standardised DALY rate, latest available year
latest_year = panel["year"].max()
latest = panel[panel["year"] == latest_year]

fig, ax = plot_choropleth(
    latest,
    column="daly_rate",
    title=f"Age-Standardised DALY Rate per 100,000 ({latest_year})",
    cmap="YlOrRd",
    legend_label="DALYs per 100k",
)
save_figure(fig, "fig01_daly_world_map")
plt.show()

## Figure 2: LISA Cluster Map

In [ ]:
# Figure 2: Local Indicators of Spatial Association (LISA) cluster map
fig, ax = plot_lisa_map(
    latest,
    column="daly_rate",
    title=f"LISA Cluster Map: DALY Rate ({latest_year})",
)
save_figure(fig, "fig02_lisa_cluster_map")
plt.show()

## Figure 3: Life Expectancy Trends by Region

In [ ]:
# Figure 3: Multi-line chart of life expectancy by WHO region
regional_le = (
    panel.groupby(["year", "who_region"])["life_expectancy"]
    .mean()
    .reset_index()
)

fig, ax = plot_line_chart(
    regional_le,
    x="year",
    y="life_expectancy",
    hue="who_region",
    title="Life Expectancy Trends by WHO Region",
    ylabel="Life Expectancy (years)",
)
save_figure(fig, "fig03_le_trends_by_region")
plt.show()

## Figure 4: SHAP Feature Importance

In [ ]:
# Figure 4: SHAP beeswarm plot for the main prediction model
fig, ax = plot_shap_beeswarm(
    model_name="xgboost_daly",
    max_display=20,
    title="SHAP Feature Importance: DALY Rate Model",
)
save_figure(fig, "fig04_shap_beeswarm")
plt.show()

## Figure 5: PAF Stacked Bar by Region

In [ ]:
# Figure 5: Population Attributable Fraction stacked bar chart by region
fig, ax = plot_stacked_bar(
    panel,
    group_col="who_region",
    value_cols=["paf_smoking", "paf_obesity", "paf_air_pollution",
                "paf_alcohol", "paf_physical_inactivity"],
    labels=["Smoking", "Obesity", "Air Pollution",
            "Alcohol", "Physical Inactivity"],
    title="Population Attributable Fraction by Risk Factor and Region",
    ylabel="PAF (%)",
)
save_figure(fig, "fig05_paf_stacked_bar")
plt.show()

## Figure 6: Sankey Attribution Diagram

In [ ]:
# Figure 6: Sankey diagram showing risk factor -> disease -> outcome attribution
fig = interactive_sankey(
    panel,
    title="Risk Factor Attribution: Sankey Diagram",
)
# Save interactive as HTML, static as PNG
out_dir = Path("..") / "reports" / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
fig.write_html(str(out_dir / "fig06_sankey_attribution.html"))
fig.write_image(str(out_dir / "fig06_sankey_attribution.png"), scale=2)
print(f"  Saved: {out_dir / 'fig06_sankey_attribution.html'}")
print(f"  Saved: {out_dir / 'fig06_sankey_attribution.png'}")
fig.show()

## Figure 7: Scenario Projections (2025-2045)

In [ ]:
# Figure 7: Scenario fan charts for projected DALY rates under different policy scenarios
fig = interactive_scenario_fan(
    panel,
    scenarios=["baseline", "optimistic", "pessimistic", "accelerated_intervention"],
    target_col="daly_rate",
    horizon=(2025, 2045),
    title="DALY Rate Projections Under Policy Scenarios (2025-2045)",
)
out_dir = Path("..") / "reports" / "figures"
fig.write_html(str(out_dir / "fig07_scenario_projections.html"))
fig.write_image(str(out_dir / "fig07_scenario_projections.png"), scale=2)
print(f"  Saved: {out_dir / 'fig07_scenario_projections.html'}")
print(f"  Saved: {out_dir / 'fig07_scenario_projections.png'}")
fig.show()

## Figure 8: Efficiency Quadrant Scatter

In [ ]:
# Figure 8: Quadrant scatter - health expenditure vs outcomes
fig, ax = plot_quadrant_scatter(
    latest,
    x="health_expenditure_pct_gdp",
    y="life_expectancy",
    size="population",
    hue="who_region",
    xlabel="Health Expenditure (% of GDP)",
    ylabel="Life Expectancy (years)",
    title=f"Health System Efficiency Quadrant ({latest_year})",
    annotate_outliers=True,
)
save_figure(fig, "fig08_efficiency_quadrant")
plt.show()

## Figure 9: Bivariate Choropleth

In [ ]:
# Figure 9: Bivariate choropleth - GDP per capita vs DALY rate
fig, ax = plot_bivariate_choropleth(
    latest,
    x_col="gdp_per_capita",
    y_col="daly_rate",
    x_label="GDP per Capita",
    y_label="DALY Rate",
    title=f"Bivariate Map: GDP per Capita vs DALY Rate ({latest_year})",
)
save_figure(fig, "fig09_bivariate_choropleth")
plt.show()

## Figure 10: GHRI Radar Comparison

In [ ]:
# Figure 10: Radar chart comparing GHRI pillar scores for country groups
ghri_df = compute_ghri(latest)

country_groups = {
    "G7": ["United States", "United Kingdom", "Germany", "France", "Japan", "Canada", "Italy"],
    "BRICS": ["Brazil", "Russia", "India", "China", "South Africa"],
    "Nordic": ["Norway", "Sweden", "Denmark", "Finland", "Iceland"],
}

fig = interactive_ghri_radar(ghri_df, country_groups=country_groups)
out_dir = Path("..") / "reports" / "figures"
fig.write_html(str(out_dir / "fig10_ghri_radar.html"))
fig.write_image(str(out_dir / "fig10_ghri_radar.png"), scale=2)
print(f"  Saved: {out_dir / 'fig10_ghri_radar.html'}")
print(f"  Saved: {out_dir / 'fig10_ghri_radar.png'}")
fig.show()

## Figure 11: Bump Chart - Country LE Rank Evolution

In [ ]:
# Figure 11: Bump chart showing life expectancy rank evolution over time
highlight_countries = [
    "United States", "China", "Japan", "Germany", "Brazil",
    "India", "Nigeria", "Norway", "South Korea", "United Kingdom",
    "South Africa", "Australia",
]

fig, ax = plot_bump_chart(
    panel,
    rank_col="life_expectancy",
    country_col="country",
    year_col="year",
    highlight=highlight_countries,
    title="Life Expectancy Rank Evolution",
    ascending=False,
)
save_figure(fig, "fig11_le_bump_chart")
plt.show()

## Export All to reports/figures/

In [ ]:
# Summary of all saved figures
from pathlib import Path

fig_dir = Path("..") / "reports" / "figures"
if fig_dir.exists():
    files = sorted(fig_dir.iterdir())
    print(f"Total figures saved: {len(files)}")
    print("=" * 60)
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s} {size_kb:8.1f} KB")
else:
    print("No figures directory found. Run all cells above first.")